# Backups y Recuperación

Backups automáticos de DB y recuperación ante fallos

## Introducción

Un backup que no se ha probado es solo un archivo. La verdadera utilidad de un backup se confirma cuando puedes restaurar exitosamente. Un backup sin prueba de restauración es arriesgado: cuando más lo necesitas, puede fallar.

### Objetivos de Aprendizaje

- Entender qué confirma que un backup es útil
- Crear scripts de backup automatizados para PostgreSQL y SQLite
- Configurar rotación de backups (retención por días)
- Implementar verificación automática de backups
- Planificar recuperación ante desastres (disaster recovery)
- Monitorear el estado de los backups

## Qué confirma que un backup es útil

> Un backup es útil cuando se ha probado su restauración. Un archivo de backup existe no significa que funcione. La única forma de confirmar la utilidad de un backup es restaurarlo exitosamente en un entorno de prueba.

In [ ]:
print("=== Qué hace útil a un backup ===")

caracteristicas = {
    "Restauración probada": "Se ha verificado que el backup se puede restaurar",
    "Frecuencia adecuada": "Se crea con la frecuencia necesaria según el RPO",
    "Retention policy": "Se eliminan backups antiguos automáticamente",
    "Almacenamiento seguro": "Está en ubicación separada del servidor original",
    "Monitoreado": "Se verifica su estado regularmente",
}

for clave, valor in caracteristicas.items():
    print(f"  {clave}: {valor}")

print("""
Un backup sin prueba de restauración es un riesgo:
- Puede estar corrupto
- El procedimiento de restore puede no funcionar
- El tiempo de recuperación puede ser mayor al esperado

La utilidad se confirma SOLO con una restauración exitosa verificada.
""")

## Scripts de Backup Automatizados

> Scripts que crean backups automáticamente, los almacenan con fecha, y limpian backups antiguos según una política de retención.

In [ ]:
backup_script = """
#!/bin/bash
# Archivo: /opt/scripts/backup.sh

BACKUP_DIR="/var/backups/mysql"
RETENTION_DAYS=7
DATE=$(date +%Y%m%d_%H%M%S)
DB_NAME="miapp_db"
DB_USER="backup_user"
DB_PASS="/opt/secrets/db_backup.pass"

mkdir -p $BACKUP_DIR

# Crear backup
mysqldump --defaults-extra-file=$DB_PASS \
    --single-transaction \
    --routines \
    --triggers \
    $DB_NAME | gzip > $BACKUP_DIR/${DB_NAME}_${DATE}.sql.gz

# Verificar que el archivo no está vacío
if [ -s "$BACKUP_DIR/${DB_NAME}_${DATE}.sql.gz" ]; then
    echo "Backup creado: ${DB_NAME}_${DATE}.sql.gz"
else
    echo "ERROR: Backup está vacío"
    exit 1
fi

# Eliminar backups antiguos (más de $RETENTION_DAYS días)
find $BACKUP_DIR -name "*.sql.gz" -mtime +$RETENTION_DAYS -delete

echo "Backup completado. Archivos actuales:"
ls -lh $BACKUP_DIR/
"""

print("=== Script de Backup ===")
print(backup_script)

print("\nEjecutar backup:")
print("  chmod +x /opt/scripts/backup.sh")
print("  /opt/scripts/backup.sh")

## Rotación de Backups (Retención)

> La política de retención define cuántos backups mantener y por cuánto tiempo. Común: diario (7 días), semanal (4 semanas), mensual (12 meses). Script que limpia automáticamente los antiguos.

In [ ]:
print("=== Política de Retención ===")

politica = {
    "Diarios": "Últimos 7 días",
    "Semanales": "Últimas 4 semanas",
    "Mensuales": "Últimos 12 meses",
    "Anuales": "Últimos 3-7 años (según regulación)",
}

for tipo, retencion in politica.items():
    print(f"  {tipo}: {retencion}")

cleanup_script = """
# Eliminar backups diarios de más de 7 días
find /var/backups/daily -name "*.sql.gz" -mtime +7 -delete

# Eliminar backups semanales de más de 4 semanas
find /var/backups/weekly -name "*.sql.gz" -mtime +28 -delete

# Eliminar backups mensuales de más de 12 meses
find /var/backups/monthly -name "*.sql.gz" -mtime +365 -delete
"""

print("\nScript de limpieza:")
print(cleanup_script)

## Verificación Automática de Backups

> Un workflow que nightly restaura el backup en un entorno de staging y verifica que los datos son consistentes. Solo así confirmas que el backup es útil.

In [ ]:
verify_workflow = """
#!/bin/bash
# Archivo: /opt/scripts/verify_backup.sh

BACKUP_FILE=$1
TEST_DB="backup_test_db"

echo "Verificando backup: $BACKUP_FILE"

# Verificar que el archivo existe y no está vacío
if [ ! -f "$BACKUP_FILE" ]; then
    echo "ERROR: Archivo no existe"
    exit 1
fi

SIZE=$(stat -f%z "$BACKUP_FILE" 2>/dev/null || stat -c%s "$BACKUP_FILE")
if [ "$SIZE" -lt 1000 ]; then
    echo "ERROR: Archivo muy pequeño ($SIZE bytes)"
    exit 1
fi

# Crear base de datos de test
mysql -e "DROP DATABASE IF EXISTS $TEST_DB; CREATE DATABASE $TEST_DB;"

# Restaurar en base de datos de test
gunzip < "$BACKUP_FILE" | mysql $TEST_DB

# Verificar que las tablas principales existen
TABLES=$(mysql -N -e "SELECT COUNT(*) FROM information_schema.tables WHERE table_schema='$TEST_DB'")
if [ "$TABLES" -lt 1 ]; then
    echo "ERROR: No se restauraron tablas"
    exit 1
fi

# Verificar conteo de registros
for table in usuarios pedidos productos; do
    COUNT=$(mysql -N -e "SELECT COUNT(*) FROM ${TEST_DB}.$table" 2>/dev/null)
    if [ -z "$COUNT" ]; then
        echo "ERROR: Tabla $table no existe"
        exit 1
    fi
    echo "  Tabla $table: $COUNT registros"
done

# Limpiar base de test
mysql -e "DROP DATABASE $TEST_DB;"

echo "VERIFICADO: Backup restaurado exitosamente"
exit 0
"""

print("=== Verificación de Backup ===")
print(verify_workflow)

print("\nEjecutar verificación:")
print("  /opt/scripts/verify_backup.sh /var/backups/daily/db_20240115.sql.gz")

## Recuperación Ante Desastres (Disaster Recovery)

> Plan de recuperación: documenta paso a paso cómo restaurar el servicio completo desde cero, incluyendo tiempo estimado (RTO) y punto máximo de pérdida de datos aceptable (RPO).

In [ ]:
print("=== Plan de Recuperación Ante Desastres ===")

plan = {
    "RTO (Recovery Time Objective)": "Tiempo máximo para restaurar: 4 horas",
    "RPO (Recovery Point Objective)": "Pérdida de datos máxima aceptable: 1 hora",
    "Último backup verificado": "2024-01-15 03:00 UTC",
    "Ubicación backups": "S3 bucket: s3://miapp-backups/",
}

for clave, valor in plan.items():
    print(f"  {clave}: {valor}")

pasos_dr = """
PASOS DE RECUPERACIÓN:

1. Provisionar nuevo servidor desde snapshot
2. Instalar Docker, nginx, dependencias
3. Clonar repositorio de aplicación
4. Restaurar latest backup de S3:
   aws s3 sync s3://miapp-backups/latest/ /var/backups/
   gunzip < backup.sql.gz | mysql db
5. Verificar aplicación responde
6. Actualizar DNS si cambió la IP
7. Verificar logs por errores

TIEMPO ESTIMADO TOTAL: 2-4 horas
"""

print(pasos_dr)

## Monitoreo del Estado de Backups

> Un dashboard o script que verifica que los backups se están creando, tienen tamaño correcto, y fueron verificados.

In [ ]:
import datetime

def check_backup_status(backup_dir, retention_days):
    print(f"=== Estado de Backups ===")
    print(f"Directorio: {backup_dir}")
    print(f"Retención: {retention_days} días\n")
    
    fecha_hoy = datetime.date.today()
    
    backups = [
        {"nombre": "db_20240115.sql.gz", "fecha": datetime.date(2024, 1, 15), "tamano": "150MB", "verificado": True},
        {"nombre": "db_20240114.sql.gz", "fecha": datetime.date(2024, 1, 14), "tamano": "148MB", "verificado": True},
        {"nombre": "db_20240113.sql.gz", "fecha": datetime.date(2024, 1, 13), "tamano": "145MB", "verificado": False},
    ]
    
    print("Backups encontrados:")
    for b in backups:
        dias_diff = (fecha_hoy - b["fecha"]).days
        estado_verif = "OK" if b["verificado"] else "SIN VERIFICAR"
        print(f"  {b['nombre']}: {b['tamano']}, {dias_diff}d atrás, verificado={estado_verif}")
    
    print("\nAlertas:")
    if not any(b["fecha"] == fecha_hoy for b in backups):
        print("  ALERTA: No hay backup de hoy")
    if not all(b["verificado"] for b in backups):
        print("  ALERTA: Hay backups sin verificar")

check_backup_status("/var/backups/daily", 7)

## Tips y Mejores Prácticas

> Un backup sin restaurar NO es un backup — es solo un archivo. Programa restauraciones de prueba al menos semanalmente.

> El 3-2-1 backup rule: 3 copias, en 2 medios diferentes, 1 fuera del sitio (cloud storage).

> Automatiza TODO: creación, rotación, verificación y alertas. Backups manuales se olvidan.

> Verifica el tamaño del backup: si es mucho menor al anterior, puede indicar un problema.

> Documenta el RTO y RPO y asegúrate de que la frecuencia de backup cumple el RPO.

## Errores Comunes

### Backup sin verificar

¿Por qué ocurre?
- Se crea el backup pero nunca se restaura para probar que funciona.

Solución
- Automatiza verificación nightly con restore a base de datos de test.

### Backups en el mismo servidor que la base de datos

¿Por qué ocurre?
- Si el disco falla, se pierden tanto la DB como los backups.

Solución
- Usa S3, GCS, o servidor de backups separado. Principio 3-2-1.

### No tener política de retención

¿Por qué ocurre?
- El disco se llena con backups de años, afectando al servidor.

Solución
- Script automático que elimine backups más antiguos que N días.

### No documentar el procedimiento de restore

¿Por qué ocurre?
- En un desastre, no hay tiempo para buscar cómo restaurar.

Solución
- Documenta paso a paso el proceso de recuperación y pratica anualmente.